In [19]:
import pandas as pd
import numpy as np
import os
import json

In [20]:
# 读取当前目录下的sample.tsv 
df = pd.read_csv('sample.tsv', sep='\t')
df.head()

,Sample Number,Sample ID,Run,Series ID,Study Title,Species,Strain,Model,Library Strategy,Sample Name,Carbon Sources,Temperature
0,F01001001,GSM675519,SRR101401,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Illumina Genome Analyzer II,RNA-Seq,Barley_34C_rep1,Barley,34°
1,F01001002,GSM675520,SRR101402,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Illumina Genome Analyzer II,RNA-Seq,Barley_45C_rep1,Barley,45°
2,F01001003,GSM675521,SRR101403,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Illumina Genome Analyzer II,RNA-Seq,Glucose_45C_rep1,Glucose,45°
3,F01001004,GSM675522,SRR101404,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Illumina Genome Analyzer II,RNA-Seq,Alfalfa_45C_rep1,Alfalfa,45°
4,F01001005,GSM675523,SRR101405,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Illumina Genome Analyzer II,RNA-Seq,Barley_45C_rep1,Barley,45°


In [21]:
# df保留Sample Number、Series ID、Study Title、Species、Strain、Sample Name、Canbon Sources、Temperature列
df_reduce = df[['Sample Number', 'Series ID', 'Study Title', 'Species', 'Strain', 'Sample Name', 'Carbon Sources', 'Temperature']]

# 将Sample Number这一列中的F开头的号码，保留一行，删除其他行
df_reduce = df_reduce[df_reduce['Sample Number'].str.startswith('F')].drop_duplicates(subset=['Sample Number'])

# 删除Sample Name这一列中的_rep1、_R1、-1、_1
df_reduce['Sample Name'] = df_reduce['Sample Name'].str.replace('_rep1', '').str.replace('_rep2','').str.replace('_R1', '').str.replace('-1', '').str.replace('_1', '')
df_reduce.head()

,Sample Number,Series ID,Study Title,Species,Strain,Sample Name,Carbon Sources,Temperature
0,F01001001,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Barley_34C,Barley,34°
1,F01001002,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Barley_45C,Barley,45°
2,F01001003,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Glucose_45C,Glucose,45°
3,F01001004,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Alfalfa_45C,Alfalfa,45°
4,F01001005,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Barley_45C,Barley,45°


In [22]:
# df_reduce新建一列，列名为Conditon，值为Strain + Carbon Sources + Temperature
df_reduce['Condition'] = df_reduce['Strain']  + '_' + df_reduce['Carbon Sources'] + '_' + df_reduce['Temperature']
df_reduce.head(5)

,Sample Number,Series ID,Study Title,Species,Strain,Sample Name,Carbon Sources,Temperature,Condition
0,F01001001,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Barley_34C,Barley,34°,ATCC 42464_Barley_34°
1,F01001002,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Barley_45C,Barley,45°,ATCC 42464_Barley_45°
2,F01001003,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Glucose_45C,Glucose,45°,ATCC 42464_Glucose_45°
3,F01001004,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Alfalfa_45C,Alfalfa,45°,ATCC 42464_Alfalfa_45°
4,F01001005,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464,Barley_45C,Barley,45°,ATCC 42464_Barley_45°


In [23]:
# df_reduce取出Sample Number、Series ID、Study Title、Species、Condition列
df_1 = df_reduce[['Sample Number', 'Series ID', 'Study Title', 'Species', 'Condition']]
df_1

,Sample Number,Series ID,Study Title,Species,Condition
0,F01001001,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Barley_34°
1,F01001002,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Barley_45°
2,F01001003,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Glucose_45°
3,F01001004,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Alfalfa_45°
4,F01001005,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Barley_45°
...,...,...,...,...,...
163,F01012002,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Glucose_45°
165,F01012003,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Gal_45°
167,F01012004,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Gal_45°
169,F01012005,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Gal_45°


In [24]:
# 读取dataset.tsv
df_2 = pd.read_csv('dataset.tsv', sep='\t')
df_2 = df_2[['GEO Series', 'Publication']]
df_2.rename(columns={'GEO Series': 'Series ID'}, inplace=True)

df_2.head(20)

,Series ID,Publication
0,GSE27323,21964414.0
1,GSE183387,NaN
2,GSE184074,35257374.0
3,GSE205182,36165620.0
4,GSE214002,36691040.0
5,GSE214142,36478865.0
6,GSE110062,29843921.0
7,GSE111986,30474279.0
8,GSE114057,31078793.0
9,GSE57235,24881579.0


In [25]:
# 将df_1和df_2合并，根据Series ID列
df_3 = pd.merge(df_1, df_2, on='Series ID', how='left')

# 重置索引
df_3 = df_3.reset_index(drop=True)

# Publication转为str，删除掉Publication值为nan的行
df_3['Publication'] = df_3['Publication'].astype(str)
df_3 = df_3[df_3['Publication'] != 'nan']

# 删除Publication最后的.0
df_3['Publication'] = df_3['Publication'].str.replace('.0', '', regex=True)

df_3

,Sample Number,Series ID,Study Title,Species,Condition,Publication
0,F01001001,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Barley_34°,21964414
1,F01001002,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Barley_45°,21964414
2,F01001003,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Glucose_45°,21964414
3,F01001004,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Alfalfa_45°,21964414
4,F01001005,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,ATCC 42464_Barley_45°,21964414
...,...,...,...,...,...,...
81,F01012002,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Glucose_45°,33995328
82,F01012003,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Gal_45°,33995328
83,F01012004,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Gal_45°,33995328
84,F01012005,GSE165516,Transcriptional profiling of Myceliophthora th...,Myceliphothora thermophila,ATCC 42464_Gal_45°,33995328


In [26]:
def get_gene_exp(gene):
    """
    根据基因id，返回该基因在各个文献对应样本的TPM表达量
    """

    # 读取exp_fname.tsv和sample_condition.csv，分别为基因的表达量(重复样本取平均值)和基因对应的文献与样本信息
    df_exp_fname = pd.read_csv('exp_fname.tsv', sep='\t')
    df_condition = pd.read_csv('sample_condition.csv', sep=',')

    # 根据gene id从df_exp_fname中取出对应的行，并且将TPM标准化为log2(x + 1)
    df = df_exp_fname[df_exp_fname['Gene id'] == gene].copy()
    df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)

    # df_condition的Sample Number列中的值对应着df_exp_fname的列名；
    # 给df_condition新添加一列，列名为Sample Exp，值为df_exp_fname中Fname号列名对应的TPM值
    df_condition['Sample Exp'] = df_condition['Sample Number'].map(df.set_index('Gene id').iloc[:, 1:].T.to_dict()[gene])

    # df_condition返回的是一个dataframe；构造字典
    output_dict = {
        'gene_id': {
            'gene_id': gene
        }
    }
    for _, row in df_condition.iterrows():
        pmid = str(row['Publication'])
        if pmid not in output_dict['gene_id']:
            output_dict['gene_id'][pmid] = {
                'PMID': pmid,
                'Study Title': row['Study Title'],
                'Species': row['Species']
            }
        sample_number = row['Sample Number']
        output_dict['gene_id'][pmid][sample_number] = {
            'Sample Number': sample_number,
            'Condition': row['Condition'],
            'Sample Exp': str(row['Sample Exp'])
        }

    # # output_dict转为json格式
    # output_json = json.dumps(output_dict, indent=4)

    return output_dict

dict_gene_exp = get_gene_exp('MYCTH_2293963')
dict_gene_exp

{'gene_id': {'gene_id': 'MYCTH_2293963',
  '21964414': {'PMID': '21964414',
   'Study Title': 'Transcriptomic response to substrate and temperature in two thermophilic fungi, Myceliophthora thermophila and Thielavia terrestris, and a related mesophile, Chaetomium globosum',
   'Species': 'Myceliphothora thermophila',
   'F01001001': {'Sample Number': 'F01001001',
    'Condition': 'ATCC 42464_Barley_34°',
    'Sample Exp': '3.64'},
   'F01001002': {'Sample Number': 'F01001002',
    'Condition': 'ATCC 42464_Barley_45°',
    'Sample Exp': '2.22'},
   'F01001003': {'Sample Number': 'F01001003',
    'Condition': 'ATCC 42464_Glucose_45°',
    'Sample Exp': '4.07'},
   'F01001004': {'Sample Number': 'F01001004',
    'Condition': 'ATCC 42464_Alfalfa_45°',
    'Sample Exp': '3.82'},
   'F01001005': {'Sample Number': 'F01001005',
    'Condition': 'ATCC 42464_Barley_45°',
    'Sample Exp': '3.09'},
   'F01001006': {'Sample Number': 'F01001006',
    'Condition': 'ATCC 42464_Glucose_34°',
    'Samp

In [27]:
#将output_dict 转为json文件
with open('gene_exp.json', 'w') as f:
    json.dump(dict_gene_exp, f, indent=4)
